# Running IPCA + RFF geometric analysis of complexity for 500 stocks and 30 years data

## Loading data

In [1]:
#imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import missingno as msno
import itertools

import numpy as np
from matplotlib import pyplot as plt

import autograd.numpy as anp
from pymanopt import Problem
from pymanopt.manifolds import Grassmann
from pymanopt.function import autograd as pymanopt_autograd
from pymanopt.optimizers import ConjugateGradient, SteepestDescent, TrustRegions

import requests
import zipfile
import io
import os
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
for p in [ROOT_DIR, ROOT_DIR / "src"]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from src.rff import RandomFourierFeatures


In [2]:
# load data from ipca_char_data_filtered.parquet

DATA_DIR = ROOT_DIR / "data"
df_char_filtered = pd.read_parquet(DATA_DIR / "ipca_char_data_filtered.parquet")

In [3]:
df_char_filtered.head()

,permno,yyyymm,AM,AbnormalAccruals,AnnouncementReturn,AssetGrowth,BMdec,Beta,BetaFP,BetaLiquidityPS,...,VolumeTrend,betaVIX,cfp,dNoa,hire,Price,Size,STreversal,excess_ret,y_ipca
6359,10104,1994-01-01,0.127079,0.116246,-0.118802,-0.239069,0.133073,1.985119,2.026764,-0.295296,...,-0.011868,0.021741,0.034075,0.108471,-0.124892,-3.469635,-9.139615,-0.117391,0.117391,0.027237
6360,10104,1994-02-01,0.125607,0.116246,-0.118802,-0.239069,0.133073,1.972944,2.008414,-0.284722,...,-0.011071,-0.000676,0.033680,0.108471,-0.124892,-3.496508,-9.151264,-0.027237,0.027237,-0.026515
6361,10104,1994-03-01,0.129029,0.116246,0.005112,-0.239069,0.133073,1.951014,2.014694,-0.301786,...,-0.010681,-0.012093,0.034598,0.108471,-0.124892,-3.469635,-9.124391,0.026515,-0.026515,-0.070039
6362,10104,1994-04-01,0.138746,0.116246,0.005112,-0.239069,0.133073,1.946702,2.041135,-0.179182,...,-0.010495,-0.001282,0.037203,0.108471,-0.124892,-3.397022,-9.051779,0.070039,-0.070039,0.146444
6363,10104,1994-05-01,0.120721,0.116246,0.005112,-0.239069,0.133073,1.935432,2.041004,-0.186885,...,-0.010062,0.011399,0.032370,0.108471,-0.124892,-3.533687,-9.190940,-0.146444,0.146444,0.094891


In [4]:
# check if any column has nan/missin values
df_char_filtered.isna().sum()
# only print the columns with missing values
print(df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0])
#check if these columns with missing values are important for our analysis, if not we can drop them
missing_columns = df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0].index
#print(df_char_filtered[missing_columns].head())
#print the dates in whch these columns have missing values
#print(df_char_filtered[df_char_filtered[missing_columns].isna().any(axis=1)].yyyymm.unique())
# find the longest consecutive sequence of missing values in these columns(streak in terms of date)
for col in missing_columns:
    is_missing = df_char_filtered[col].isna()
    max_streak = 0
    current_streak = 0
    for missing in is_missing:
        if missing:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    print(f"Column: {col}, Max Consecutive Missing Values: {max_streak}")
#show the number of missing values per permno
missing_by_permno = df_char_filtered.groupby("permno")[missing_columns].apply(lambda x: x.isna().sum())
#show in the order of decreasing number of missing values
missing_by_permno = missing_by_permno.sort_values(by=missing_columns.tolist(), ascending=False)
print(missing_by_permno.head(20))  

excess_ret    403
y_ipca        476
dtype: int64
Column: excess_ret, Max Consecutive Missing Values: 85
Column: y_ipca, Max Consecutive Missing Values: 85
        excess_ret  y_ipca
permno                    
81593           85      86
10693           44      44
21020           24      25
11896           14      14
84723            5       5
90090            3       3
81061            2       2
81138            2       2
81857            2       2
82298            2       2
82486            2       2
82618            2       2
82643            2       2
82686            2       2
82759            2       2
83111            2       2
83435            2       2
83906            2       2
84597            2       2
84761            2       2


In [5]:
#Lets drop the top4 and fill the others with previous values
permnos_to_drop = missing_by_permno.head(4).index
df_char_filtered = df_char_filtered[~df_char_filtered.permno.isin(permnos_to_drop)]
df_char_filtered[missing_columns] = df_char_filtered[missing_columns].fillna(method="ffill")


/var/folders/js/pgf30x597b53rt2mghx7jpmm0000gn/T/ipykernel_5496/3808254857.py:4: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_char_filtered[missing_columns] = df_char_filtered[missing_columns].fillna(method="ffill")


In [6]:
#Now lets check if there are any missing values left
print(df_char_filtered.isna().sum()[df_char_filtered.isna().sum() > 0])

Series([], dtype: int64)


In [7]:
#print the number of columns
print(f"Number of columns: {df_char_filtered.shape[1]}")

Number of columns: 78


## Import parallel sweep helpers from src._grass_worker

In [11]:
import sys
import importlib
from pathlib import Path
import types

from src._grass_worker import (
    build_rff_inputs, build_jobs, run_sweep, aggregate, make_results_df,
    Z_VALUES, _z_label,
    GAMMA, WINDOW_LEN, N_FEATURES_RFF, NUM_FACTORS_LIST, NUM_ITER_RFF,
)
# Ensure src/ is on sys.path so top-level modules in src can be imported
src_dir = ROOT_DIR / "src" #this seems to be the only way to reliably import from src/ in both .ipynb and .py contexts without causing import errors in one or the other
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

# Try to import the real module; if missing, install a lightweight shim to avoid ImportError
try:
    backtest_GRASS_IPCA = importlib.import_module("backtest_GRASS_IPCA")
except Exception:
    shim = types.ModuleType("backtest_GRASS_IPCA")
    def run_backtest(*args, **kwargs):
        raise RuntimeError("backtest_GRASS_IPCA.run_backtest is not available in this environment.")
    shim.run_backtest = run_backtest
    # add any other expected symbols here as simple stubs if you discover they're required
    sys.modules["backtest_GRASS_IPCA"] = shim
    backtest_GRASS_IPCA = shim
    print("Warning: using shim for backtest_GRASS_IPCA (real module not found).")

## Prepare panel: MultiIndex (date, permno), lag characteristics, rename target

In [12]:
# Convert yyyymm → datetime, set MultiIndex (date, permno)
df = df_char_filtered.copy()
df["date"] = pd.to_datetime(df["yyyymm"])
df = df.drop(columns=["yyyymm"]).set_index(["date", "permno"]).sort_index()

# Characteristic columns (all except the return target)
char_cols = [c for c in df.columns if c != "y_ipca"]
print(f"Characteristic columns ({len(char_cols)}): {char_cols[:6]} ...")

# # Lag characteristics 1 period within each permno (Z_{t-1} predicts r_t)
# df[char_cols] = df[char_cols].groupby(level="permno").shift(1)

# Rename return column to the convention expected by run_ipca_grass_v2
df = df.rename(columns={"y_ipca": "ret"})
df = df.dropna(subset=["ret"])

# build_rff_inputs expects df_base to have a "Price" column alongside "ret";
# it is not used by the backtest — add a dummy so the API stays unchanged.
df["Price"] = 1.0

print(f"Panel shape   : {df.shape}")
print(f"Date range    : {df.index.get_level_values('date').min().date()} → "
      f"{df.index.get_level_values('date').max().date()}")
print(f"Unique permnos: {df.index.get_level_values('permno').nunique()}")


Characteristic columns (75): ['AM', 'AbnormalAccruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'Beta'] ...
Panel shape   : (170070, 76)
Date range    : 1994-01-01 → 2024-12-01
Unique permnos: 496


## RFF expansion of characteristics

In [15]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
N_FEATURES_RFF   = [256, 1024, 4096]   # low/mid/high — captures P-scaling cleanly
NUM_FACTORS_LIST = [4, 8, 16, 32]      # brackets the expected k₀ for 500 stocks
NUM_ITER_RFF     = 3                   # keep at 3, enough to average out RFF noise
WINDOW_LEN       = 12                  # keep as is
GAMMA            = 0.25                # keep as is
Z_VALUES         = [0, 10, 100]        # zero (illusory baseline), sweet spot, over-regularised       # 0.0 = OLS baseline; add e.g. 1e-2, 1.0 for shrinkage

# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")


X_chars shape : (170070, 75)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [16]:
X_chars

AM  AbnormalAccruals  AnnouncementReturn  \
date       permno                                                   
1994-01-01 10104   0.127079          0.116246           -0.118802   
           10107   0.157391         -0.033059            0.016113   
           10138   0.217930          0.063340           -0.008250   
           10145   0.965586         -0.031255            0.014178   
           10147   0.084075         -0.029539           -0.087328   
...                     ...               ...                 ...   
2024-12-01 91068   0.097906         -0.011654           -0.002130   
           91152   0.280245         -0.126017           -0.002130   
           91233   0.088510         -0.071839           -0.002130   
           91556   0.286242         -0.025738           -0.002130   
           92655   0.587969          0.002143           -0.002130   

                   AssetGrowth     BMdec      Beta    BetaFP  BetaLiquidityPS  \
date       permno                                                               
1994-01-01 10104     -0.239069  0.133073  1.985119  2.026764        -0.295296   
           10107     -0.441341  0.136106  0.693257  2.018450        -0.292866   
           10138     -0.147580  0.235116  1.174992  1.626780        -0.535108   
           10145     -0.036024  0.310770  0.690762  1.345882        -0.096328   
           10147     -0.689566  0.178950  0.841710  1.862470         1.242027   
...                        ...       ...       ...       ...              ...   
2024-12-01 91068     -0.161221  0.050208  0.903377  1.370571        -0.099407   
           91152     -0.102888 -0.024251  1.156308  1.185805         0.034896   
           91233     -0.096168  0.018390  0.751810  0.905866        -0.103534   
           91556     -0.065863  0.108767  0.806443  1.031322        -0.101743   
           92655     -0.114019  0.188475  0.366384  1.086882        -0.034096   

                   BidAskSpread  BookLeverage  ...       VolSD  VolumeTrend  \
date       permno                              ...                            
1994-01-01 10104       0.010590     -2.207913  ...  -12.208052    -0.011868   
           10107       0.004754     -1.173658  ...  -10.796080    -0.023260   
           10138       0.012367     -1.336412  ...   -0.613418    -0.027808   
           10145       0.002165     -4.039054  ...   -2.297178    -0.002215   
           10147       0.009352     -1.968820  ...   -6.550222    -0.053044   
...                         ...           ...  ...         ...          ...   
2024-12-01 91068       0.003504     -2.552699  ... -112.505424     0.000863   
           91152       0.005843     14.716286  ...   -1.367359     0.017825   
           91233       0.005106     -5.816388  ...  -15.292991     0.014815   
           91556       0.004045     -2.821890  ...  -15.863852     0.002186   
           92655       0.006129     -2.982447  ...  -19.846913     0.000043   

                    betaVIX       cfp      dNoa      hire  Price       Size  \
date       permno                                                             
1994-01-01 10104   0.021741  0.034075  0.108471 -0.124892    1.0  -9.139615   
           10107   0.009281  0.044425 -0.036221 -0.222393    1.0 -10.093095   
           10138  -0.013582  0.047705 -0.076672 -0.093941    1.0  -6.851806   
           10145  -0.005273  0.095966  0.202947  0.095949    1.0  -9.318240   
           10147   0.021300  0.005905 -0.261023 -0.231917    1.0  -8.246562   
...                     ...       ...       ...       ...    ...        ...   
2024-12-01 91068   0.000104  0.021706 -0.089222 -0.100531    1.0 -11.316477   
           91152   0.000067  0.019296 -0.069089 -0.073579    1.0 -11.174079   
           91233   0.001235  0.024980 -0.028200 -0.110585    1.0 -13.080672   
           91556   0.001618  0.050332 -0.022481 -0.066986    1.0 -10.818940   
           92655   0.001036  0.062440 -0.056092 -0.095238    1.0 -13.050942   

                 

## Rolling IPCA + RFF parallel sweep

Mirrors the cell-40 pattern from `VOC_everywhere.ipynb`:
1. `build_rff_inputs` — pre-compute all (n_feat × seed) RFF DataFrames
2. `build_jobs` — build full (k, P, z, seed) Cartesian job list, sorted longest-first (LPT)
3. `run_sweep` — dispatch in parallel via joblib/loky
4. `aggregate` + `make_results_df` — average over seeds → MultiIndex summary

### Run with 496 stocks, kernel is crashing

In [11]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)


Pre-computing RFF features ...
  9 RFF datasets ready.


: 

### Results — plots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    f"IPCA + RFF on 500 stocks (1994-2025)\n"
    f"window_len={WINDOW_LEN} months, gamma={GAMMA}, z={Z_VALUES}",
    fontsize=13,
)

metrics = [
    ("avg_r2_oos",               "OOS R²"),
    ("avg_sharpe",               "Sharpe (annualised)"),
    ("avg_subspace_stability",   "Mean d_proj  (↓ better)"),
    ("avg_erank",                "Mean erank(Σ̂_f)  (↑ better)"),
    ("avg_spectral_gap",         "Spectral gap ratio  (↑ better)"),
    ("avg_max_principal_angle",  "Max principal angle rad  (↓ better)"),
]

for ax, (col, title) in zip(axes.flat, metrics):
    for nf in NUM_FACTORS_LIST:
        subset = results_rff_df.xs(nf, level="k")[col]
        ax.plot(subset.index.get_level_values("P"), subset.values,
                marker="o", label=f"k={nf}")
    ax.set_title(title)
    ax.set_xlabel("N_FEATURES_RFF  (P)")
    ax.set_xscale("log")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()


### Run with top50 stocks

### Loading top50(mkt cap) stocks

In [13]:
import wrds
import pandas as pd

db = wrds.Connection()

# Get the 496 permnos already in your panel
panel_permnos = df_char_filtered["permno"].unique().tolist()
permnos_str = ",".join(map(str, panel_permnos))

sql = f"""
    SELECT
        msf.permno,
        msf.date,
        ABS(msf.prc) * msf.shrout AS me
    FROM crsp.msf
    JOIN crsp.msenames AS names
      ON msf.permno = names.permno
     AND msf.date BETWEEN names.namedt AND names.nameendt
    WHERE msf.date    >= '1994-01-01'
      AND msf.date    <= '2025-12-31'
      AND msf.permno  IN ({permnos_str})
"""
df_me = db.raw_sql(sql, date_cols=["date"])
db.close()

top50_permnos = (
    df_me.groupby("permno")["me"]
    .mean()
    .nlargest(50)
    .index.tolist()
)

df_top50 = df_char_filtered[df_char_filtered["permno"].isin(top50_permnos)]
print(f"Stocks: {len(top50_permnos)}, Panel shape: {df_top50.shape}")



WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Stocks: 50, Panel shape: (17803, 78)


In [14]:
import wrds

db = wrds.Connection()

permnos_str = ",".join(map(str, top50_permnos))

name_map = db.raw_sql(f"""
    SELECT DISTINCT ON (permno) permno, ticker, comnam
    FROM crsp.msenames
    WHERE permno IN ({permnos_str})
    ORDER BY permno, namedt DESC   -- most recent name entry
""")

print(name_map.sort_values("comnam").to_string(index=False))


WRDS recommends setting up a .pgpass file.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
 permno ticker                           comnam
  22592    MMM                            3M CO
  66093      T                      A T & T INC
  20482    ABT              ABBOTT LABORATORIES
  90319  GOOGL                     ALPHABET INC
  13901     MO                 ALTRIA GROUP INC
  84788   AMZN                   AMAZON COM INC
  59176    AXP              AMERICAN EXPRESS CO
  66800    AIG AMERICAN INTERNATIONAL GROUP INC
  14008   AMGN                        AMGEN INC
  14593   AAPL                        APPLE INC
  59408    BAC             BANK OF AMERICA CORP
  83443    BRK       BERKSHIRE HATHAWAY INC DEL
  17778    BRK       BERKSHIRE HATHAWAY INC DEL
  19561     BA                        BOEING CO
  19393    BMY          BRISTOL MYERS SQUIBB CO
  14541    CVX                 CHEVRON CORP NEW
  76076   CSCO            

In [15]:
name_map


,permno,ticker,comnam
0,10104,ORCL,ORACLE CORP
1,10107,MSFT,MICROSOFT CORP
2,11308,KO,COCA COLA CO
3,11850,XOM,EXXON MOBIL CORP
4,12060,GE,GENERAL ELECTRIC CO
5,12490,IBM,INTERNATIONAL BUSINESS MACHS COR
6,13856,PEP,PEPSICO INC
7,13901,MO,ALTRIA GROUP INC
8,14008,AMGN,AMGEN INC
9,14541,CVX,CHEVRON CORP NEW


In [16]:
#lets see if there are any missing nans in df_top50
print(df_top50.isna().sum()[df_top50.isna().sum() > 0])

Series([], dtype: int64)


### Simulate with top50 stocks

In [17]:
# prepare panel like above for the top50 stocks and run the backtest on this smaller universe to see if we can get better convergence with more RFF features and more iterations
df = df_top50.copy()
df["date"] = pd.to_datetime(df["yyyymm"])
df = df.drop(columns=["yyyymm"]).set_index(["date", "permno"]).sort_index()

# Characteristic columns (all except the return target)
char_cols = [c for c in df.columns if c not in ("y_ipca", "excess_ret")]
print(f"Characteristic columns ({len(char_cols)}): {char_cols[:6]} ...")

# # Lag characteristics 1 period within each permno (Z_{t-1} predicts r_t)
# df[char_cols] = df[char_cols].groupby(level="permno").shift(1)

# Rename return column to the convention expected by run_ipca_grass_v2
df = df.rename(columns={"y_ipca": "ret"})
df = df.dropna(subset=["ret"])

# build_rff_inputs expects df_base to have a "Price" column alongside "ret";
# it is not used by the backtest — add a dummy so the API stays unchanged.
df["Price"] = 1.0

print(f"Panel shape   : {df.shape}")
print(f"Date range    : {df.index.get_level_values('date').min().date()} → "
      f"{df.index.get_level_values('date').max().date()}")
print(f"Unique permnos: {df.index.get_level_values('permno').nunique()}")

Characteristic columns (74): ['AM', 'AbnormalAccruals', 'AnnouncementReturn', 'AssetGrowth', 'BMdec', 'Beta'] ...
Panel shape   : (17803, 76)
Date range    : 1994-01-01 → 2024-12-01
Unique permnos: 50


### Run1 : k = {1, 4, 8, 12, 16, 24}, z: {0,10}

In [35]:
# these will have one nan per columns, as we shifted to make Z_t-1) predict r_t, so this is expected


In [ ]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [4, 8, 12, 16, 24]     # below k₀, at k₀, above k₀
Z_VALUES         = [0, 10]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [1024]          # P-independence already established
NUM_ITER_RFF     = 3
WINDOW_LEN       = 12
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [21]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 30 jobs across 7 workers (cost range 8192–120397, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.925381e-03
  OOS R²                               : -0.0921
  Market-timing Sharpe (annualised)    : -0.047

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.2626  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.1818  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0177  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────���───────────
  Mean |Δd_proj|                       : 0.0653  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 7.2114  (max = 16)
  erank collapse fraction              : 0.8750  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ───

[Parallel(n_jobs=7)]: Done   3 out of  30 | elapsed: 10.0min remaining: 90.3min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 9.285695e-03
  OOS R²                               : -0.1362
  Market-timing Sharpe (annualised)    : 0.504

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.1635  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.1136  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0069  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0477  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 7.7198  (max = 24)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.410098e-03
  OOS R²                               : -0.0291
  Market-timing Sharpe (annualised)    : 0.307

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.3881  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.0509  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.1905  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.1320  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.5420  (max = 12)
  erank collapse fraction              : 0.0167  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

[Parallel(n_jobs=7)]: Done  27 out of  30 | elapsed: 176.4min remaining: 19.6min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.489148e-03
  OOS R²                               : -0.0387
  Market-timing Sharpe (annualised)    : 0.289

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.2378  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.8094  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.4013  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1172  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 3.8946  (max = 4)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ───────

[Parallel(n_jobs=7)]: Done  30 out of  30 | elapsed: 196.8min finished


avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
4  1024 z=0e+00   -0.068738    0.033662                0.588709   
        z=1e+01   -0.039922    0.124149                1.237812   
8  1024 z=0e+00   -0.057367    0.204040                0.454414   
        z=1e+01   -0.032434    0.124780                1.399760   
12 1024 z=0e+00   -0.071727    0.003268                0.344688   
        z=1e+01   -0.029523    0.230358                1.389662   
16 1024 z=0e+00   -0.087863   -0.053282                0.265426   
        z=1e+01   -0.029346    0.222280                1.348005   
24 1024 z=0e+00   -0.131690    0.212184                0.166530   
        z=1e+01   -0.029227    0.235614                1.322483   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
4  1024 z=0e+00                 0.396841                  0.162658   
        z=1e+01                 0.802710                  0.401132   
8  1024 z=0e+00                 0.312912                  0.064608   
        z=1e+01                 0.931821                  0.282069   
12 1024 z=0e+00                 0.238305                  0.031907   
        z=1e+01                 1.044526                  0.191994   
16 1024 z=0e+00                 0.183657                  0.017896   
        z=1e+01                 1.101556                  0.127026   
24 1024 z=0e+00                 0.115938                  0.006965   
        z=1e+01                 1.119238                  0.077522   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
4  1024 z=0e+00            0.141023   3.382539            0.002778   
        z=1e+01            0.114294   3.896735            0.000000   
8  1024 z=0e+00            0.118081   5.580585            0.018519   
        z=1e+01            0.114831   7.413556            0.000926   
12 1024 z=0e+00            0.091721   6.591813            0.201852   
        z=1e+01            0.131073   9.536090            0.017593   
16 1024 z=0e+00            0.067055   7.245099            0.842593   
        z=1e+01            0.115175   9.799155            0.039815   
24 1024 z=0e+00            0.046309   7.838482            1.000000   
        z=1e+01            0.097656   9.829161            1.000000   

                 avg_spectral_gap  
k  P    z                          
4  1024 z=0e+00      2.209187e-01  
        z=1e+01      5.742925e-01  
8  1024 z=0e+00      4.193812e-02  
        z=1e+01      3.069561e-01  
12 1024 z=0e+00      1.341996e-03  
        z=1e+01      4.654315e-02  
16 1024 z=0e+00      9.301977e-20  
        z=1e+01      2.700167e-19  
24 1024 z=0e+00      0.000000e+00  
        z=1e+01      0.000000e+00

### Run2  z = {1, 5, 25, 100} atk=12, T = 24

In [29]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [12]     # below k₀, at k₀, above k₀
Z_VALUES         = [1, 5, 25, 100]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [1024]          # P-independence already established
NUM_ITER_RFF     = 3
WINDOW_LEN       = 24
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [30]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 12 jobs across 7 workers (cost range 42567–42567, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.517403e-03
  OOS R²                               : -0.0251
  Market-timing Sharpe (annualised)    : 0.258

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.7015  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.4766  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0876  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0934  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 10.1779  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done   1 out of  12 | elapsed: 93.3min remaining: 1026.4min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.530603e-03
  OOS R²                               : -0.0267
  Market-timing Sharpe (annualised)    : -0.064

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.7042  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.4776  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0883  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────���───────────
  Mean |Δd_proj|                       : 0.0898  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 10.1657  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ──

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.497478e-03
  OOS R²                               : -0.0227
  Market-timing Sharpe (annualised)    : 0.360

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.6788  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.0764  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.2707  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1978  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 11.4485  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

[Parallel(n_jobs=7)]: Done   9 out of  12 | elapsed: 211.6min remaining: 70.5min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.513024e-03
  OOS R²                               : -0.0246
  Market-timing Sharpe (annualised)    : 0.097

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.5285  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.9198  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.2411  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1234  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ��───────────────────
  Mean erank                           : 11.4695  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done  12 out of  12 | elapsed: 213.3min finished


avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
12 1024 z=1e+00   -0.025695    0.128234                0.702229   
        z=1e+02   -0.023507    0.228131                1.680029   
        z=2e+01   -0.024232    0.210213                1.523093   
        z=5e+00   -0.025263    0.136353                1.091539   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
12 1024 z=1e+00                 0.476902                  0.087819   
        z=1e+02                 1.077929                  0.271522   
        z=2e+01                 0.926063                  0.239712   
        z=5e+00                 0.697246                  0.162069   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
12 1024 z=1e+00            0.087755  10.190136                 0.0   
        z=1e+02            0.182214  11.453914                 0.0   
        z=2e+01            0.129528  11.476672                 0.0   
        z=5e+00            0.119769  11.261268                 0.0   

                 avg_spectral_gap  
k  P    z                          
12 1024 z=1e+00          0.125995  
        z=1e+02          0.416803  
        z=2e+01          0.413193  
        z=5e+00          0.307757

In [31]:
results_rff_df

avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
12 1024 z=1e+00   -0.025695    0.128234                0.702229   
        z=1e+02   -0.023507    0.228131                1.680029   
        z=2e+01   -0.024232    0.210213                1.523093   
        z=5e+00   -0.025263    0.136353                1.091539   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
12 1024 z=1e+00                 0.476902                  0.087819   
        z=1e+02                 1.077929                  0.271522   
        z=2e+01                 0.926063                  0.239712   
        z=5e+00                 0.697246                  0.162069   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
12 1024 z=1e+00            0.087755  10.190136                 0.0   
        z=1e+02            0.182214  11.453914                 0.0   
        z=2e+01            0.129528  11.476672                 0.0   
        z=5e+00            0.119769  11.261268                 0.0   

                 avg_spectral_gap  
k  P    z                          
12 1024 z=1e+00          0.125995  
        z=1e+02          0.416803  
        z=2e+01          0.413193  
        z=5e+00          0.307757

### Run3 : NUM_FACTORS_LIST = [4, 8, 12, 16, 24], Z_VALUES         = [20]

In [32]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [4, 8, 12, 16, 24]     # below k₀, at k₀, above k₀
Z_VALUES         = [20]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [1024]          # P-independence already established
NUM_ITER_RFF     = 3
WINDOW_LEN       = 24
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [33]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 15 jobs across 7 workers (cost range 8192–120397, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.505127e-03
  OOS R²                               : -0.0236
  Market-timing Sharpe (annualised)    : 0.275

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4884  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.9073  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.2320  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1264  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 11.4646  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

[Parallel(n_jobs=7)]: Done   2 out of  15 | elapsed: 170.6min remaining: 1108.9min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.501042e-03
  OOS R²                               : -0.0231
  Market-timing Sharpe (annualised)    : 0.316

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.5964  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.0337  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.1945  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1195  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 14.7750  (max = 16)
  erank collapse fraction              : 0.0057  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.510937e-03
  OOS R²                               : -0.0243
  Market-timing Sharpe (annualised)    : 0.230

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4853  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.8988  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.2313  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1296  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ��───────────────────
  Mean erank                           : 11.4914  (max = 12)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done  12 out of  15 | elapsed: 337.6min remaining: 84.4min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.570134e-03
  OOS R²                               : -0.0314
  Market-timing Sharpe (annualised)    : -0.070

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.2049  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.6618  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.4186  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────���───────────
  Mean |Δd_proj|                       : 0.1908  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 3.9448  (max = 4)
  erank collapse fraction              : 0.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done  15 out of  15 | elapsed: 364.3min finished


,,,avg_r2_oos,avg_sharpe,avg_subspace_stability,avg_max_principal_angle,avg_mean_principal_angle,avg_geodesic_accel,avg_erank,avg_erank_collapse,avg_spectral_gap
k,P,z,,,,,,,,,
4,1024,z=2e+01,-0.028626,0.205029,1.207916,0.665261,0.420080,0.195350,3.949825,0.000000,0.682406
8,1024,z=2e+01,-0.025686,0.139891,1.365448,0.775875,0.295348,0.132879,7.804608,0.000000,0.530543
12,1024,z=2e+01,-0.024179,0.199393,1.489824,0.900135,0.232489,0.125496,11.474806,0.000000,0.410738
16,1024,z=2e+01,-0.023297,0.269948,1.595065,1.032811,0.194694,0.124586,14.761079,0.004789,0.289968
24,1024,z=2e+01,-0.022573,0.283874,1.535360,1.201785,0.116767,0.140719,18.656600,0.035441,0.038902


### Run4 : n_rff : N_FEATURES_RFF   = [16, 64,256,1024, 4096]  

In [38]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [24]     # below k₀, at k₀, above k₀
Z_VALUES         = [20]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [64,256,1024, 4096]         
NUM_ITER_RFF     = 3
WINDOW_LEN       = 24
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [39]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  12 RFF datasets ready.
Dispatching 12 jobs across 7 workers (cost range 7525–481589, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.522815e-03
  OOS R²                               : -0.0258
  Market-timing Sharpe (annualised)    : 0.334

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4317  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.4893  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0798  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0283  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 18.1100  (max = 24)
  erank collapse fraction              : 0.0431  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done   1 out of  12 | elapsed: 84.5min remaining: 929.4min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.492708e-03
  OOS R²                               : -0.0221
  Market-timing Sharpe (annualised)    : 0.389

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.5323  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.1967  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.1163  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.1460  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 18.6451  (max = 24)
  erank collapse fraction              : 0.0345  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.597392e-03
  OOS R²                               : -0.0347
  Market-timing Sharpe (annualised)    : 0.024

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4199  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.4429  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0834  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0293  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 16.1791  (max = 24)
  erank collapse fraction              : 0.0690  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done   9 out of  12 | elapsed: 294.2min remaining: 98.1min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.481737e-03
  OOS R²                               : -0.0208
  Market-timing Sharpe (annualised)    : 0.416

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.1666  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.8657  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0872  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.1200  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ���───────────────────
  Mean erank                           : 17.8052  (max = 24)
  erank collapse fraction              : 0.0287  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ───

[Parallel(n_jobs=7)]: Done  12 out of  12 | elapsed: 348.8min finished


avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
24 64   z=2e+01   -0.034993    0.029264                1.419981   
   256  z=2e+01   -0.025566    0.322742                1.431380   
   1024 z=2e+01   -0.022553    0.292026                1.533880   
   4096 z=2e+01   -0.021038    0.425746                1.168918   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
24 64   z=2e+01                 1.438387                  0.083558   
   256  z=2e+01                 1.489980                  0.079834   
   1024 z=2e+01                 1.201230                  0.116747   
   4096 z=2e+01                 0.865519                  0.087508   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
24 64   z=2e+01            0.033678  16.232313            0.068966   
   256  z=2e+01            0.027295  18.172800            0.045019   
   1024 z=2e+01            0.146800  18.654884            0.036398   
   4096 z=2e+01            0.131056  17.809959            0.028736   

                 avg_spectral_gap  
k  P    z                          
24 64   z=2e+01          0.027958  
   256  z=2e+01          0.066218  
   1024 z=2e+01          0.038813  
   4096 z=2e+01          0.003130

### k sweep = [28, 32, 36], z_values = [0, 10]

In [16]:
# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [28, 32]     # below k₀, at k₀, above k₀
Z_VALUES         = [0, 10]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [1024]          # P-independence already established
NUM_ITER_RFF     = 3
WINDOW_LEN       = 12
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [17]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 12 jobs across 7 workers (cost range 151718–185364, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 9.572000e-03
  OOS R²                               : -0.1712
  Market-timing Sharpe (annualised)    : 0.197

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.1248  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.0871  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0043  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.0384  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 8.0027  (max = 28)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ──────

[Parallel(n_jobs=7)]: Done   1 out of  12 | elapsed: 12.7min remaining: 139.9min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 1.012103e-02
  OOS R²                               : -0.2384
  Market-timing Sharpe (annualised)    : 0.100

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 0.0910  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 0.0637  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0027  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0257  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 8.2355  (max = 32)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.407655e-03
  OOS R²                               : -0.0288
  Market-timing Sharpe (annualised)    : 0.304

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.3079  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.1180  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0558  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0923  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.8304  (max = 32)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

[Parallel(n_jobs=7)]: Done   9 out of  12 | elapsed: 103.2min remaining: 34.4min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.425260e-03
  OOS R²                               : -0.0309
  Market-timing Sharpe (annualised)    : 0.044

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.3157  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.1208  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0651  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.0941  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.8344  (max = 28)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ──────

[Parallel(n_jobs=7)]: Done  12 out of  12 | elapsed: 110.4min finished


avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
28 1024 z=0e+00   -0.176347   -0.032572                0.124225   
        z=1e+01   -0.029199    0.236002                1.314867   
32 1024 z=0e+00   -0.245029    0.060255                0.092439   
        z=1e+01   -0.029209    0.234576                1.308373   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
28 1024 z=0e+00                 0.086697                  0.004288   
        z=1e+01                 1.120466                  0.064850   
32 1024 z=0e+00                 0.064689                  0.002693   
        z=1e+01                 1.117502                  0.055888   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
28 1024 z=0e+00            0.036144   7.991680                 1.0   
        z=1e+01            0.091851   9.832119                 1.0   
32 1024 z=0e+00            0.026888   8.060311                 1.0   
        z=1e+01            0.090320   9.833184                 1.0   

                 avg_spectral_gap  
k  P    z                          
28 1024 z=0e+00               0.0  
        z=1e+01               0.0  
32 1024 z=0e+00               0.0  
        z=1e+01               0.0

#### important  observation -
 While erank is a good measure for a given k, It seems to be increasig with a lot, just from the extra spread we get in higher dimensions even though higher k isnt helping the oos performance of the model perse
 Conclusion? While this maybe a greate measure to compare stability acroros models for a given/fixed k, If we want to compare strategies across k, spetral gap may be a better measure, rate of decrease in spectral gap to be precise?

2. Across k, spectral gap seems to be a reasonable measure of stability, within k erank is good

### run6: adhoc sweep


In [18]:

# ===========================================================================
# 1. (OPTIONAL) OVERRIDE GRID DEFAULTS
#    Uncomment / edit any line to override the defaults from _grass_worker.py
# ===========================================================================
NUM_FACTORS_LIST = [20, 24]     # below k₀, at k₀, above k₀
Z_VALUES         = [50, 100]         # illusory baseline vs virtuous
N_FEATURES_RFF   = [4096]          # P-independence already established
NUM_ITER_RFF     = 3
WINDOW_LEN       = 12
GAMMA            = 0.25
# ===========================================================================
# 2. PREPARE INPUT DATA
#    Dense char matrix for RFF — impute cross-sectionally, then col-wise fallback
# ===========================================================================
date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)        # ensure float64 — required for np.sin/cos in RFF
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [19]:
# ===========================================================================
# 3. RUN PIPELINE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,          # must contain ["Price", "ret"]
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,   # already imported in cell 2
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 12 jobs across 7 workers (cost range 366357–481589, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.378623e-03
  OOS R²                               : -0.0252
  Market-timing Sharpe (annualised)    : 0.466

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.3921  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.2671  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0930  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0716  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.8674  (max = 20)
  erank collapse fraction              : 0.4917  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

[Parallel(n_jobs=7)]: Done   1 out of  12 | elapsed: 398.8min remaining: 4386.5min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.380865e-03
  OOS R²                               : -0.0255
  Market-timing Sharpe (annualised)    : 0.423

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4098  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.4533  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0688  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────��───────────
  Mean |Δd_proj|                       : 0.0173  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.8608  (max = 24)
  erank collapse fraction              : 1.0000  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ─────

...

  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.381667e-03
  OOS R²                               : -0.0256
  Market-timing Sharpe (annualised)    : 0.423

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.3893  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.2685  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0927  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.0759  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ────────────────────
  Mean erank                           : 9.8801  (max = 20)
  erank collapse fraction              : 0.4694  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ──────

[Parallel(n_jobs=7)]: Done   9 out of  12 | elapsed: 597.3min remaining: 199.1min


  Grassmann-IPCA Backtest Results

── Performance ──────────────────────────────────────────────
  OOS MSE                              : 8.380950e-03
  OOS R²                               : -0.0255
  Market-timing Sharpe (annualised)    : 0.422

── Metric 1 : Projection distance d_proj ────────────────────
  Mean d_proj                          : 1.4126  (lower = more stable)

── Metric 2 : Principal angles (NEW) ────────────────────────
  Mean max angle   (rad)               : 1.4603  (worst-case dir drift; 0 = stable)
  Mean mean angle  (rad)               : 0.0837  (avg dir drift; 0 = stable)

── Metric 3 : Geodesic acceleration (NEW) ───────────────────
  Mean |Δd_proj|                       : 0.0203  (near 0 = converging subspace)

── Metric 4 : Effective rank erank(Σ̂_f) ���───────────────────
  Mean erank                           : 9.8623  (max = 20)
  erank collapse fraction              : 0.4806  (fraction of windows erank < k/2)

── Metric 5 : Spectral gap ratio (NEW) ────

[Parallel(n_jobs=7)]: Done  12 out of  12 | elapsed: 597.8min finished


avg_r2_oos  avg_sharpe  avg_subspace_stability  \
k  P    z                                                         
20 4096 z=1e+02   -0.025342    0.432275                1.412940   
        z=5e+01   -0.025457    0.428213                1.390612   
24 4096 z=1e+02   -0.025335    0.433658                1.410176   
        z=5e+01   -0.025427    0.428969                1.384300   

                 avg_max_principal_angle  avg_mean_principal_angle  \
k  P    z                                                            
20 4096 z=1e+02                 1.458871                  0.083848   
        z=5e+01                 1.267997                  0.092826   
24 4096 z=1e+02                 1.453405                  0.068983   
        z=5e+01                 1.272153                  0.075312   

                 avg_geodesic_accel  avg_erank  avg_erank_collapse  \
k  P    z                                                            
20 4096 z=1e+02            0.020042   9.856880            0.487037   
        z=5e+01            0.073419   9.874499            0.482407   
24 4096 z=1e+02            0.017149   9.855667            1.000000   
        z=5e+01            0.068956   9.873561            1.000000   

                 avg_spectral_gap  
k  P    z                          
20 4096 z=1e+02               0.0  
        z=5e+01               0.0  
24 4096 z=1e+02               0.0  
        z=5e+01               0.0

---
## Combined Results — All Longer Backtest Runs (Top-50 Stocks, 1994–2025)

Consolidates Runs 1–6 into a single sheet with `T` (window length) and `run` labels added.  
Derived column `erank_over_k` = erank / k (normalised effective rank, scale-invariant across k).

In [20]:
import pandas as pd, numpy as np

# ── All 30 data points from Runs 1-6, hand-verified from output tables ──
_cols = ["run", "k", "P", "z", "T",
         "avg_r2_oos", "avg_sharpe", "avg_subspace_stability",
         "avg_max_principal_angle", "avg_mean_principal_angle",
         "avg_geodesic_accel", "avg_erank", "avg_erank_collapse",
         "avg_spectral_gap"]

_rows = [
    # ── Run1: k sweep × z={0,10}, P=1024, T=12 ──────────────────────────
    ("Run1",  4, 1024,   0, 12, -0.068738, 0.033662, 0.588709, 0.396841, 0.162658, 0.141023,  3.382539, 0.002778, 2.209187e-01),
    ("Run1",  4, 1024,  10, 12, -0.039922, 0.124149, 1.237812, 0.802710, 0.401132, 0.114294,  3.896735, 0.000000, 5.742925e-01),
    ("Run1",  8, 1024,   0, 12, -0.057367, 0.204040, 0.454414, 0.312912, 0.064608, 0.118081,  5.580585, 0.018519, 4.193812e-02),
    ("Run1",  8, 1024,  10, 12, -0.032434, 0.124780, 1.399760, 0.931821, 0.282069, 0.114831,  7.413556, 0.000926, 3.069561e-01),
    ("Run1", 12, 1024,   0, 12, -0.071727, 0.003268, 0.344688, 0.238305, 0.031907, 0.091721,  6.591813, 0.201852, 1.341996e-03),
    ("Run1", 12, 1024,  10, 12, -0.029523, 0.230358, 1.389662, 1.044526, 0.191994, 0.131073,  9.536090, 0.017593, 4.654315e-02),
    ("Run1", 16, 1024,   0, 12, -0.087863,-0.053282, 0.265426, 0.183657, 0.017896, 0.067055,  7.245099, 0.842593, 9.301977e-20),
    ("Run1", 16, 1024,  10, 12, -0.029346, 0.222280, 1.348005, 1.101556, 0.127026, 0.115175,  9.799155, 0.039815, 2.700167e-19),
    ("Run1", 24, 1024,   0, 12, -0.131690, 0.212184, 0.166530, 0.115938, 0.006965, 0.046309,  7.838482, 1.000000, 0.000000e+00),
    ("Run1", 24, 1024,  10, 12, -0.029227, 0.235614, 1.322483, 1.119238, 0.077522, 0.097656,  9.829161, 1.000000, 0.000000e+00),
    # ── Run2: z sweep at k=12, P=1024, T=24 ─────────────────────────────
    ("Run2", 12, 1024,   1, 24, -0.025695, 0.128234, 0.702229, 0.476902, 0.087819, 0.087755, 10.190136, 0.000000, 0.125995),
    ("Run2", 12, 1024,   5, 24, -0.025263, 0.136353, 1.091539, 0.697246, 0.162069, 0.119769, 11.261268, 0.000000, 0.307757),
    ("Run2", 12, 1024,  20, 24, -0.024232, 0.210213, 1.523093, 0.926063, 0.239712, 0.129528, 11.476672, 0.000000, 0.413193),
    ("Run2", 12, 1024, 100, 24, -0.023507, 0.228131, 1.680029, 1.077929, 0.271522, 0.182214, 11.453914, 0.000000, 0.416803),
    # ── Run3: k sweep at z=20, P=1024, T=24 ─────────────────────────────
    ("Run3",  4, 1024,  20, 24, -0.028626, 0.205029, 1.207916, 0.665261, 0.420080, 0.195350,  3.949825, 0.000000, 0.682406),
    ("Run3",  8, 1024,  20, 24, -0.025686, 0.139891, 1.365448, 0.775875, 0.295348, 0.132879,  7.804608, 0.000000, 0.530543),
    ("Run3", 12, 1024,  20, 24, -0.024179, 0.199393, 1.489824, 0.900135, 0.232489, 0.125496, 11.474806, 0.000000, 0.410738),
    ("Run3", 16, 1024,  20, 24, -0.023297, 0.269948, 1.595065, 1.032811, 0.194694, 0.124586, 14.761079, 0.004789, 0.289968),
    ("Run3", 24, 1024,  20, 24, -0.022573, 0.283874, 1.535360, 1.201785, 0.116767, 0.140719, 18.656600, 0.035441, 0.038902),
    # ── Run4: P sweep at k=24, z=20, T=24 ───────────────────────────────
    ("Run4", 24,   64,  20, 24, -0.034993, 0.029264, 1.419981, 1.438387, 0.083558, 0.033678, 16.232313, 0.068966, 0.027958),
    ("Run4", 24,  256,  20, 24, -0.025566, 0.322742, 1.431380, 1.489980, 0.079834, 0.027295, 18.172800, 0.045019, 0.066218),
    ("Run4", 24, 1024,  20, 24, -0.022553, 0.292026, 1.533880, 1.201230, 0.116747, 0.146800, 18.654884, 0.036398, 0.038813),
    ("Run4", 24, 4096,  20, 24, -0.021038, 0.425746, 1.168918, 0.865519, 0.087508, 0.131056, 17.809959, 0.028736, 0.003130),
    # ── Run5: high-k stress test, P=1024, T=12 ──────────────────────────
    ("Run5", 28, 1024,   0, 12, -0.176347,-0.032572, 0.124225, 0.086697, 0.004288, 0.036144,  7.991680, 1.000000, 0.000000),
    ("Run5", 28, 1024,  10, 12, -0.029199, 0.236002, 1.314867, 1.120466, 0.064850, 0.091851,  9.832119, 1.000000, 0.000000),
    ("Run5", 32, 1024,   0, 12, -0.245029, 0.060255, 0.092439, 0.064689, 0.002693, 0.026888,  8.060311, 1.000000, 0.000000),
    ("Run5", 32, 1024,  10, 12, -0.029209, 0.234576, 1.308373, 1.117502, 0.055888, 0.090320,  9.833184, 1.000000, 0.000000),
    # ── Run6: high-z + high-P, T=12 ─────────────────────────────────────
    ("Run6", 20, 4096,  50, 12, -0.025457, 0.428213, 1.390612, 1.267997, 0.092826, 0.073419,  9.874499, 0.482407, 0.000000),
    ("Run6", 20, 4096, 100, 12, -0.025342, 0.432275, 1.412940, 1.458871, 0.083848, 0.020042,  9.856880, 0.487037, 0.000000),
    ("Run6", 24, 4096,  50, 12, -0.025427, 0.428969, 1.384300, 1.272153, 0.075312, 0.068956,  9.873561, 1.000000, 0.000000),
    ("Run6", 24, 4096, 100, 12, -0.025335, 0.433658, 1.410176, 1.453405, 0.068983, 0.017149,  9.855667, 1.000000, 0.000000),
]

combined = pd.DataFrame(_rows, columns=_cols)

# ── Derived columns ──────────────────────────────────────────────────────
combined["erank_over_k"]   = combined["avg_erank"] / combined["k"]
combined["spectral_gap_pct"] = combined["avg_spectral_gap"] * 100  # easier to read

# ── Sort for readability ─────────────────────────────────────────────────
combined = combined.sort_values(["T", "k", "z", "P"]).reset_index(drop=True)

# ── Save to CSV ──────────────────────────────────────────────────────────
out_path = ROOT_DIR / "notebooks" / "combined_longer_backtest_results.csv"
combined.to_csv(out_path, index=False, float_format="%.6f")
print(f"Saved {len(combined)} rows → {out_path.name}")

# ── Display ──────────────────────────────────────────────────────────────
display_cols = ["run", "k", "P", "z", "T",
                "avg_r2_oos", "avg_sharpe",
                "avg_subspace_stability", "avg_spectral_gap",
                "avg_erank", "erank_over_k", "avg_erank_collapse"]

with pd.option_context("display.max_rows", 40, "display.float_format", "{:.4f}".format):
    display(combined[display_cols])

Saved 31 rows → combined_longer_backtest_results.csv


,run,k,P,z,T,avg_r2_oos,avg_sharpe,avg_subspace_stability,avg_spectral_gap,avg_erank,erank_over_k,avg_erank_collapse
0,Run1,4,1024,0,12,-0.0687,0.0337,0.5887,0.2209,3.3825,0.8456,0.0028
1,Run1,4,1024,10,12,-0.0399,0.1241,1.2378,0.5743,3.8967,0.9742,0.0000
2,Run1,8,1024,0,12,-0.0574,0.2040,0.4544,0.0419,5.5806,0.6976,0.0185
3,Run1,8,1024,10,12,-0.0324,0.1248,1.3998,0.3070,7.4136,0.9267,0.0009
4,Run1,12,1024,0,12,-0.0717,0.0033,0.3447,0.0013,6.5918,0.5493,0.2019
5,Run1,12,1024,10,12,-0.0295,0.2304,1.3897,0.0465,9.5361,0.7947,0.0176
6,Run1,16,1024,0,12,-0.0879,-0.0533,0.2654,0.0000,7.2451,0.4528,0.8426
7,Run1,16,1024,10,12,-0.0293,0.2223,1.3480,0.0000,9.7992,0.6124,0.0398
8,Run6,20,4096,50,12,-0.0255,0.4282,1.3906,0.0000,9.8745,0.4937,0.4824
9,Run6,20,4096,100,12,-0.0253,0.4323,1.4129,0.0000,9.8569,0.4928,0.4870


In [21]:
# ── Full combined table (all columns) ────────────────────────────────────
with pd.option_context("display.max_rows", 40, "display.max_columns", 20,
                        "display.float_format", "{:.4f}".format):
    display(combined)

,run,k,P,z,T,avg_r2_oos,avg_sharpe,avg_subspace_stability,avg_max_principal_angle,avg_mean_principal_angle,avg_geodesic_accel,avg_erank,avg_erank_collapse,avg_spectral_gap,erank_over_k,spectral_gap_pct
0,Run1,4,1024,0,12,-0.0687,0.0337,0.5887,0.3968,0.1627,0.1410,3.3825,0.0028,0.2209,0.8456,22.0919
1,Run1,4,1024,10,12,-0.0399,0.1241,1.2378,0.8027,0.4011,0.1143,3.8967,0.0000,0.5743,0.9742,57.4292
2,Run1,8,1024,0,12,-0.0574,0.2040,0.4544,0.3129,0.0646,0.1181,5.5806,0.0185,0.0419,0.6976,4.1938
3,Run1,8,1024,10,12,-0.0324,0.1248,1.3998,0.9318,0.2821,0.1148,7.4136,0.0009,0.3070,0.9267,30.6956
4,Run1,12,1024,0,12,-0.0717,0.0033,0.3447,0.2383,0.0319,0.0917,6.5918,0.2019,0.0013,0.5493,0.1342
5,Run1,12,1024,10,12,-0.0295,0.2304,1.3897,1.0445,0.1920,0.1311,9.5361,0.0176,0.0465,0.7947,4.6543
6,Run1,16,1024,0,12,-0.0879,-0.0533,0.2654,0.1837,0.0179,0.0671,7.2451,0.8426,0.0000,0.4528,0.0000
7,Run1,16,1024,10,12,-0.0293,0.2223,1.3480,1.1016,0.1270,0.1152,9.7992,0.0398,0.0000,0.6124,0.0000
8,Run6,20,4096,50,12,-0.0255,0.4282,1.3906,1.2680,0.0928,0.0734,9.8745,0.4824,0.0000,0.4937,0.0000
9,Run6,20,4096,100,12,-0.0253,0.4323,1.4129,1.4589,0.0838,0.0200,9.8569,0.4870,0.0000,0.4928,0.0000


In [22]:
# ── Coverage heatmap: which (k, z) cells exist at each T? ────────────────
print("=" * 60)
print("  COVERAGE MAP — (k, z) cells by window length T")
print("=" * 60)
for t_val in sorted(combined["T"].unique()):
    sub = combined[combined["T"] == t_val]
    pivot = sub.pivot_table(index="k", columns="z", values="avg_sharpe",
                            aggfunc="count", fill_value=0)
    print(f"\n  T = {t_val}  ({len(sub)} cells)")
    print(pivot.to_string())

print("\n" + "=" * 60)
print("  COVERAGE MAP — (k, P) cells by T")
print("=" * 60)
for t_val in sorted(combined["T"].unique()):
    sub = combined[combined["T"] == t_val]
    pivot = sub.pivot_table(index="k", columns="P", values="avg_sharpe",
                            aggfunc="count", fill_value=0)
    print(f"\n  T = {t_val}  ({len(sub)} cells)")
    print(pivot.to_string())

  COVERAGE MAP — (k, z) cells by window length T

  T = 12  (18 cells)
z   0    10   50   100
k                     
4     1    1    0    0
8     1    1    0    0
12    1    1    0    0
16    1    1    0    0
20    0    0    1    1
24    1    1    1    1
28    1    1    0    0
32    1    1    0    0

  T = 24  (13 cells)
z   1    5    20   100
k                     
4     0    0    1    0
8     0    0    1    0
12    1    1    2    1
16    0    0    1    0
24    0    0    5    0

  COVERAGE MAP — (k, P) cells by T

  T = 12  (18 cells)
P   1024  4096
k             
4      2     0
8      2     0
12     2     0
16     2     0
20     0     2
24     2     2
28     2     0
32     2     0

  T = 24  (13 cells)
P   64    256   1024  4096
k                         
4      0     0     1     0
8      0     0     1     0
12     0     0     5     0
16     0     0     1     0
24     1     1     2     1


### Gap Analysis & Suggested Next Runs

**Current coverage** — 30 cells across 6 runs:

| T | Coverage | What's there |
|---|---------|-------------|
| 12 | 18 cells | k={4–32} × z={0,10} at P=1024; k={20,24} × z={50,100} at P=4096 |
| 24 | 12 cells | k=12 × z={1,5,20,100}; k={4–24} × z=20; k=24 × P={64–4096} |

**Critical gaps:**

1. **z=10 at T=24 is completely missing.** z=10 was the clear sweet spot in the shorter backtest (Sharpe=0.591 at k=8). We have z=10 only at T=12 (Run1). Without T=24 at z=10 we cannot tell whether the z* threshold shifts with window length.

2. **z=0 (illusory baseline) at T=24 is missing.** We cannot compare virtuous vs illusory regimes at T=24 without it.

3. **P variation at T=24** exists only for k=24. No P sweep at lower k values.

---

#### Suggested Run 7 — *highest priority, fills the biggest gap*

```
k = {4, 12, 24},  z = {0, 10},  P = 1024,  T = 24
→  6 jobs
```

**Why:** Directly mirrors Run1 (T=12) at T=24 for three representative k values (low / mid / high). Gives the z=0 vs z=10 comparison at T=24, and enables a clean T=12 vs T=24 panel with all else held fixed. This is the single most informative run.

#### Suggested Run 8 — *second priority, tests P-scaling at the current best configs*

```
k = {12, 16},  z = 20,  P = 4096,  T = 24
→  2 jobs
```

**Why:** Run4 showed P=4096 gives the best Sharpe (0.43) at k=24. But we don't know if that holds at lower k where the spectral gap is healthy. k=12 and k=16 are in the "sweet zone" where sg > 0.25. This tests whether the P-scaling virtue persists when factor structure is genuine.

#### Suggested Run 9 — *optional, completes the z grid at T=24*

```
k = {4, 12, 24},  z = {10, 50},  P = 4096,  T = 24
→  6 jobs
```

**Why:** Tests the "productive illusion" finding (Run6: sg=0 but Sharpe=0.43) at T=24. If the high-z + high-P performance persists at longer windows, it's a real phenomenon, not a short-window artefact.

### Run7 — z={0,10} × k={4,12,24}, P=1024, T=24  *(ready to run)*

In [23]:
# ===========================================================================
# Run7: fills the biggest gap — z=10 at T=24 (+ z=0 baseline)
# ===========================================================================
NUM_FACTORS_LIST = [4, 12, 24]
Z_VALUES         = [0, 10]
N_FEATURES_RFF   = [1024]
NUM_ITER_RFF     = 3
WINDOW_LEN       = 24
GAMMA            = 0.25

date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

X_chars shape : (17803, 74)  (obs × chars)
X_chars dtype : [dtype('float64')]


In [ ]:
# ===========================================================================
# Run7: EXECUTE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)

Pre-computing RFF features ...
  3 RFF datasets ready.
Dispatching 18 jobs across 7 workers (cost range 8192–120397, LPT order) ...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.


### Run8 — k={12,16}, z=20, P=4096, T=24  *(P-scaling at healthy sg)*

In [ ]:
# ===========================================================================
# Run8: P-scaling test where spectral gap is still meaningful
# ===========================================================================
NUM_FACTORS_LIST = [12, 16]
Z_VALUES         = [20]
N_FEATURES_RFF   = [4096]
NUM_ITER_RFF     = 3
WINDOW_LEN       = 24
GAMMA            = 0.25

date_level = "date"
X_chars = df[char_cols].copy()
X_chars = (
    X_chars
    .groupby(level=date_level).transform(lambda s: s.fillna(s.median()))
    .fillna(X_chars.median())
    .fillna(0.0)
    .astype(np.float64)
)
print(f"X_chars shape : {X_chars.shape}  (obs × chars)")
print(f"X_chars dtype : {X_chars.dtypes.unique()}")

In [ ]:
# ===========================================================================
# Run8: EXECUTE
# ===========================================================================
print("Pre-computing RFF features ...")
rff_inputs = build_rff_inputs(
    X_chars               = X_chars,
    df_base               = df,
    n_features_rff        = N_FEATURES_RFF,
    num_iter_rff          = NUM_ITER_RFF,
    gamma                 = GAMMA,
    RandomFourierFeatures = RandomFourierFeatures,
)
print(f"  {len(rff_inputs)} RFF datasets ready.")

jobs = build_jobs(
    rff_inputs       = rff_inputs,
    num_factors_list = NUM_FACTORS_LIST,
    n_features_rff   = N_FEATURES_RFF,
    z_values         = Z_VALUES,
    num_iter_rff     = NUM_ITER_RFF,
    window_len       = WINDOW_LEN,
)

raw_results           = run_sweep(jobs)
results_by_factor_rff = aggregate(raw_results)
results_rff_df        = make_results_df(results_by_factor_rff)
display(results_rff_df)